# 05 — Train/Test Strategy, Baseline & Advanced Models, Class Imbalance

Phases 8-11.

In [1]:
import sys
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('..').resolve() / 'src'))
import features, models, evaluation

df = pd.read_csv('../data/processed/transactions_features.csv')
X = df[features.MODEL_FEATURE_COLUMNS]
y = df['is_fraud']
X.shape, y.sum()

((283726, 33), np.int64(473))

## Phase 8 — Train / validation / test split

Stratified three-way split: **train** (60%) fits models, **validation** (20%) is used for model comparison and threshold selection, **test** (20%) is held out completely until final evaluation (notebook 06) so reported numbers aren't tuned to the data they're reported on.

In [2]:
X_train, X_val, X_test, y_train, y_val, y_test = models.three_way_split(X, y)
for name, yy in [('train', y_train), ('validation', y_val), ('test', y_test)]:
    print(f'{name:<12} n={len(yy):>7,}  fraud={yy.sum():>4}  fraud_rate={100*yy.mean():.4f}%')

train        n=170,235  fraud= 284  fraud_rate=0.1668%
validation   n= 56,745  fraud=  94  fraud_rate=0.1657%
test         n= 56,746  fraud=  95  fraud_rate=0.1674%


Stratification keeps the ~0.17% fraud rate consistent across all three splits — without it, a small split could easily end up with a meaningfully different (or even zero) fraud rate by chance.

## Feature scaling (for Logistic Regression only)

`V1`-`V28` are already standardized (a side effect of the original PCA), but `Amount`, `log_amount`, `hour_of_day`, etc. are not — Logistic Regression's coefficients (and solver convergence) are scale-sensitive, so it gets a `StandardScaler` fit **only on the training fold**. Random Forest and Gradient Boosting are scale-invariant and use the raw features.

In [3]:
scaler = models.make_scaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns, index=X_val.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

## Phase 9 — Baseline: Logistic Regression

The baseline exists to answer one question: *how much lift do the more complex models actually buy us?* Without it, a strong Random Forest number means nothing on its own.

In [4]:
mdls = models.get_models()
lr = mdls['Logistic Regression']
lr.fit(X_train_scaled, y_train)
val_results = {}
val_results['Logistic Regression'] = evaluation.compute_metrics(y_val, lr.predict_proba(X_val_scaled)[:, 1])
val_results['Logistic Regression']

{'threshold': 0.5,
 'precision': 0.05442176870748299,
 'recall': 0.851063829787234,
 'f1': 0.10230179028132992,
 'roc_auc': 0.9666075076325857,
 'pr_auc': 0.7054599575319468}

## Phase 10 — Advanced models: Random Forest, Gradient Boosting

In [5]:
rf = mdls['Random Forest']
rf.fit(X_train, y_train)
val_results['Random Forest'] = evaluation.compute_metrics(y_val, rf.predict_proba(X_val)[:, 1])

gb = mdls['Gradient Boosting']
gb_sample_weight = models.balanced_sample_weight(y_train)
gb.fit(X_train, y_train, sample_weight=gb_sample_weight)
val_results['Gradient Boosting'] = evaluation.compute_metrics(y_val, gb.predict_proba(X_val)[:, 1])

evaluation.comparison_table(val_results)

,Model,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Logistic Regression,0.0544,0.8511,0.1023,0.9666,0.7055
1,Random Forest,0.8875,0.7553,0.8161,0.9607,0.7760
2,Gradient Boosting,0.2179,0.8298,0.3451,0.9711,0.7155


(This comparison table is on the **validation** set, used to pick a model — the test-set version of this table, the one that counts, is in `06_model_evaluation.ipynb`.)

## Phase 11 — Class imbalance: does the balancing actually matter?

Rather than assert that `class_weight='balanced'` is necessary, this compares it against an **unweighted** Logistic Regression on the same split — the imbalance-handling choice should earn its place with a number, not be assumed.

In [6]:
from sklearn.linear_model import LogisticRegression

lr_unweighted = LogisticRegression(max_iter=1000, random_state=42)
lr_unweighted.fit(X_train_scaled, y_train)
unweighted_metrics = evaluation.compute_metrics(y_val, lr_unweighted.predict_proba(X_val_scaled)[:, 1])

imbalance_compare = evaluation.comparison_table({
    'LR - unweighted': unweighted_metrics,
    'LR - class_weight=balanced': val_results['Logistic Regression'],
})
imbalance_compare

,Model,Precision,Recall,F1,ROC-AUC,PR-AUC
0,LR - unweighted,0.8596,0.5213,0.6490,0.9695,0.6949
1,LR - class_weight=balanced,0.0544,0.8511,0.1023,0.9666,0.7055


**At the default 0.5 threshold**, the unweighted model's recall is dramatically lower — with 284:1 imbalance, a classifier can achieve very low error simply by leaning toward "legitimate" for almost everything, catching only the most obvious fraud. `class_weight` corrects for this at training time by penalizing missed fraud cases more heavily; note ROC-AUC/PR-AUC (threshold-independent, rank-based metrics) are far less affected, since both models still *rank* fraud cases similarly — it's specifically the precision/recall trade-off at a fixed threshold that balancing changes. **Threshold choice (Phase 14) matters at least as much as this**, and is treated properly with a full sweep in `06_model_evaluation.ipynb`.

### An alternative: SMOTE oversampling

Applied strictly to the training fold, after the split (see `src/models.smote_resample` docstring for why order matters here).

In [7]:
X_train_smote, y_train_smote = models.smote_resample(X_train_scaled, y_train)
print('Before SMOTE:', y_train.value_counts().to_dict())
print('After SMOTE: ', y_train_smote.value_counts().to_dict())

lr_smote = LogisticRegression(max_iter=1000, random_state=42)
lr_smote.fit(X_train_smote, y_train_smote)
smote_metrics = evaluation.compute_metrics(y_val, lr_smote.predict_proba(X_val_scaled)[:, 1])

evaluation.comparison_table({
    'LR - class_weight=balanced': val_results['Logistic Regression'],
    'LR - SMOTE': smote_metrics,
})

Before SMOTE: {0: 169951, 1: 284}
After SMOTE:  {0: 169951, 1: 169951}


,Model,Precision,Recall,F1,ROC-AUC,PR-AUC
0,LR - class_weight=balanced,0.0544,0.8511,0.1023,0.9666,0.7055
1,LR - SMOTE,0.0523,0.8511,0.0985,0.9653,0.7040


Both approaches solve the same underlying problem and land in a similar place here; `class_weight='balanced'` is used going forward (notebook 06) since it needs no extra training-data copy and applies identically to all three model families, keeping the final comparison apples-to-apples.

## Persist for 06_model_evaluation.ipynb

In [8]:
model_dir = Path('..') / 'data' / 'processed' / 'models'
model_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(scaler, model_dir / 'scaler.joblib')
joblib.dump(lr, model_dir / 'logistic_regression.joblib')
joblib.dump(rf, model_dir / 'random_forest.joblib')
joblib.dump(gb, model_dir / 'gradient_boosting.joblib')
print('Saved models to', model_dir)

Saved models to ..\data\processed\models
